# 07 — Investigación: Preprocesado Avanzado
Aplica técnicas de preprocesado financiero al **mejor modelo** de cada categoría (competición).
Compara MAE con/sin cada técnica.

Técnicas activas:
1. **StandardScaler** (fit solo en train)
2. **Diferenciación fraccional (FFD)** — `[EXTENDER]`

> Referencia teórica: `docs/resumenes/b3_s2_mapa.md`, `docs/resumenes/preprocesado-datos.md`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings, yfinance as yf
warnings.simplefilter('ignore')

from sklearn.preprocessing import StandardScaler
from keras import Sequential, Input
from keras.layers import LSTM, Dense

from utils import (TICKERS, INPUT_WINDOWS, OUTPUT_WINDOWS,
                   create_time_series_data, make_splits, eval_mae,
                   get_callbacks, compile_model,
                   plot_mae_matrix, build_results_df, best_per_window)

In [ ]:
EPOCHS     = 100
BATCH_SIZE = 64
QUICK_MODE = False
if QUICK_MODE: EPOCHS = 20

precios = yf.download(TICKERS, start='1945-01-01', auto_adjust=True, progress=False)['Close']
precios.dropna(axis=1, inplace=True)
returns_raw = np.log(precios).diff().dropna()   # log-retornos base (sin escalar)
print(f'Retornos: {returns_raw.shape}')

## Modelo de referencia
Usar el mejor modelo de la competición. Aquí usamos **lstm_s** como ejemplo.
Reemplazar por el modelo ganador identificado en `06_resultados.ipynb`.

In [ ]:
# Modelo de referencia (mismo que competición)
def build_best(V_in):
    return compile_model(Sequential([
        Input((V_in, 23)), LSTM(64), Dense(23)]))
    # [EXTENDER] Reemplazar por el modelo real ganador de 06_resultados.ipynb

## Técnica 1 — Sin preprocesado (baseline de investigación)
Mismos resultados que en la competición. Sirve de referencia.

In [ ]:
def entrenar_y_eval(returns_data, V_in, V_out, nombre):
    """Entrena build_best y devuelve dict de MAE."""
    X, y = create_time_series_data(returns_data, V_in, V_out)
    X_tr, X_v, X_ts, y_tr, y_v, y_ts = make_splits(X, y)
    model = build_best(V_in)
    model.fit(X_tr, y_tr, validation_data=(X_v, y_v),
              epochs=EPOCHS, batch_size=BATCH_SIZE,
              callbacks=get_callbacks(), verbose=0)
    return {'train': eval_mae(model, X_tr, y_tr),
            'val':   eval_mae(model, X_v,  y_v),
            'test':  eval_mae(model, X_ts, y_ts),
            'params': model.count_params()}

res_crudo = {}
for V_in in INPUT_WINDOWS:
    for V_out in OUTPUT_WINDOWS:
        res_crudo[('crudo', V_in, V_out)] = entrenar_y_eval(returns_raw, V_in, V_out, 'crudo')
        print(f'crudo  in={V_in}  out={V_out}  test={res_crudo[("crudo",V_in,V_out)]["test"]:.4f}')

## Técnica 2 — StandardScaler (fit solo en train)
Estandariza los retornos de cada activo a media=0, std=1.
**Importante**: el scaler se ajusta SOLO sobre los datos de train para evitar data leakage.

Referencia: `docs/resumenes/preprocesado-datos.md` — Estandarización.

In [ ]:
res_scaled = {}

for V_in in INPUT_WINDOWS:
    for V_out in OUTPUT_WINDOWS:
        X_raw, y_raw = create_time_series_data(returns_raw, V_in, V_out)
        X_tr, X_v, X_ts, y_tr, y_v, y_ts = make_splits(X_raw, y_raw)

        # Ajustar scaler SOLO en train (reshape temporal para StandardScaler)
        N_tr, T, F = X_tr.shape
        scaler = StandardScaler()
        X_tr_s  = scaler.fit_transform(X_tr.reshape(-1, F)).reshape(N_tr, T, F)
        X_v_s   = scaler.transform(X_v.reshape(-1, F)).reshape(len(X_v),  T, F)
        X_ts_s  = scaler.transform(X_ts.reshape(-1, F)).reshape(len(X_ts), T, F)

        model = build_best(V_in)
        model.fit(X_tr_s, y_tr, validation_data=(X_v_s, y_v),
                  epochs=EPOCHS, batch_size=BATCH_SIZE,
                  callbacks=get_callbacks(), verbose=0)

        key = ('scaled', V_in, V_out)
        res_scaled[key] = {'train':  eval_mae(model, X_tr_s, y_tr),
                           'val':    eval_mae(model, X_v_s,  y_v),
                           'test':   eval_mae(model, X_ts_s, y_ts),
                           'params': model.count_params()}
        print(f'scaled  in={V_in}  out={V_out}  test={res_scaled[key]["test"]:.4f}')

## Diferenciación Fraccional (FFD) — `[EXTENDER]`
Técnica avanzada que mantiene la memoria de la serie mientras la hace más estacionaria.
Parámetro `d` controla el balance: `d=1` es diferenciación clásica, `d=0` es la serie original.

Referencia: `docs/resumenes/b3_s2_mapa.md` — Diferenciación Fraccional.

In [ ]:
# [EXTENDER] Implementar FFD:
#
# def ffd_weights(d, size, threshold=1e-5):
#     """Calcula los pesos de la ventana FFD de orden d."""
#     w = [1.]
#     for k in range(1, size):
#         w.append(-w[-1] * (d - k + 1) / k)
#         if abs(w[-1]) < threshold: break
#     return np.array(w[::-1])
#
# def apply_ffd(series, d=0.4, threshold=1e-5):
#     """Aplica FFD a una serie 1D."""
#     w = ffd_weights(d, len(series), threshold)
#     result = np.full(len(series), np.nan)
#     for i in range(len(w)-1, len(series)):
#         result[i] = (w * series[i-len(w)+1:i+1]).sum()
#     return result
#
# # Aplicar a cada activo y comparar MAE con crudo y scaled
print('[EXTENDER] FFD no implementado en esta versión. Ver comentarios arriba.')

## Comparación: crudo vs. StandardScaler

In [ ]:
all_res = {**res_crudo, **res_scaled}
df_inv  = build_results_df(all_res)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, tecnica in zip(axes, ['crudo', 'scaled']):
    try:
        sub = df_inv.xs(tecnica, level='modelo')['test']
        mat = sub.unstack('V_out').reindex(index=INPUT_WINDOWS, columns=OUTPUT_WINDOWS)
        import seaborn as sns
        sns.heatmap(mat.astype(float), annot=True, fmt='.4f', cmap='YlOrRd_r',
                    ax=ax, linewidths=.5)
        ax.set_title(f'{tecnica} — MAE test')
        ax.set_xlabel('V_out'); ax.set_ylabel('V_in')
    except Exception:
        ax.set_title(f'{tecnica} — sin datos')
plt.suptitle('Investigación: efecto del preprocesado', fontsize=13)
plt.tight_layout(); plt.show()

print('\nDiferencia MAE (scaled - crudo) — negativo = scaled mejora:')
if res_crudo and res_scaled:
    diffs = [(k[1], k[2],
              round(res_scaled[('scaled',k[1],k[2])]['test'] - res_crudo[('crudo',k[1],k[2])]['test'], 5))
             for k in res_crudo]
    df_diff = pd.DataFrame(diffs, columns=['V_in','V_out','Δtest']).set_index(['V_in','V_out'])
    print(df_diff['Δtest'].unstack('V_out').to_string())